In [2]:
from torch import nn
import torch
import pandas as pd
import benepar, spacy
from nltk.tokenize import word_tokenize
import nltk
import ssl
import stanza
from collections import Counter
from torch.optim.lr_scheduler import OneCycleLR
import re

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
stanza.download('en')
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,constituency')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2026-06-24 02:36:04 INFO: Downloaded file to /Users/kenne/Library/Caches/stanza/1.13.0/resources/resources.json
2026-06-24 02:36:04 INFO: Downloading default packages for language: en (English) ...
2026-06-24 02:36:05 INFO: File exists: /Users/kenne/Library/Caches/stanza/1.13.0/resources/en/default.zip
2026-06-24 02:36:06 INFO: Finished downloading models and saved to /Users/kenne/Library/Caches/stanza/1.13.0/resources
2026-06-24 02:36:06 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-24 02:36:07 INFO: Downloaded file to /Users/kenne/Library/Caches/stanza/1.13.0/resources/resources.json
2026-06-24 02:36:07 WARNING: Language en package default expects mwt, which has been added
2026-06-24 02:36:07 INFO: Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| token

In [152]:
def cast_to_float(context):
    return([float(x) for x in context.strip("[]").split()])
# def clean_text(text):
#     if pd.isna(text):
#         return text

#     # Remove single quotes around a word/phrase, e.g. 'Oliver' -> Oliver
#     text = re.sub(r"'([\w\s\-]+)'", r'\1', text)

#     # Replace & with "n" in abbreviations like B&Q -> BnQ
#     # Matches a letter/digit, then &, then a letter/digit (no spaces around &)
#     text = re.sub(r'(?<=\w)&(?=\w)', 'n', text)

#     # Lowercase everything
#     text = text.lower()

#     # Add a space before and after every forward slash
#     text = re.sub(r'\s*/\s*', ' / ', text)

#     # Add a space before and after every hyphen
#     text = re.sub(r'\s*-\s*', ' - ', text)

#     # Collapse repeated ! or ? into a single one, e.g. "!!!" -> "!", "???" -> "?"
#     text = re.sub(r'!+', '!', text)
#     text = re.sub(r'\?+', '?', text)

#     return text

In [ ]:
df = pd.read_csv("paraphrase_encodings.csv")
# df = df.iloc[:-49_000]
df['context_vector'] = df['context_vector'].apply(lambda x: cast_to_float(x))
# df['source'] = df['source'].apply(clean_text)

In [206]:
model_size = len(df['context_vector'].iloc[0])
print(model_size)

384


In [ ]:
# all_tokens = []
# vocab = {}
# for sentence in (df['source'].tolist()):
#     tokens = word_tokenize(sentence)
#     all_tokens.extend(tokens)
# for token, _ in Counter(all_tokens).most_common():
#     vocab[token] = len(vocab)

# vocab_size = len(vocab)
# print(vocab_size)

37144


In [6]:
def skip_unary(node):
    """Follow unary chains down until reaching a node with 0 or 2+ children."""
    while len(node.children) == 1:
        node = node.children[0]
    return node

vocab = {}
def stanza_tree_to_nested(node, role=0, max_role = None):
    """
    Recursively convert a Stanza constituency tree node to a nested list,
    collapsing unary chains.

    Each leaf becomes:        [word, role]
    Each inner node becomes:  [[child_0, child_1, ...], role]
    where each child's own role is its index among its siblings.
    Unary chains (nodes with a single child, recursively) are skipped.

    Args:
        node: a Stanza constituency tree node
        role: the index of this node among its parent's children (default 0 for root)

    Returns:
        A nested list representation of the subtree.
    """
    if max_role is None:
        max_role = [0]  # mutable container so mutations propagate up

    # Collapse unary chains before processing
    node = skip_unary(node)

    if role > max_role[0]:
        max_role[0] = role

    # Leaf node: return [word, role]
    if len(node.children) == 0:
        label = node.label.lower()
        try:
            token = torch.tensor(vocab[label], dtype=torch.long)
        except KeyError:
            token = torch.tensor(len(vocab), dtype=torch.long)
            vocab[label] = len(vocab)
        return [token, torch.tensor(role, dtype = torch.long)], max_role[0]

    # Inner node: children go into their own sublist, then role is appended
    converted_children = [
        stanza_tree_to_nested(child, sibling_index, max_role)[0]
        for sibling_index, child in enumerate(node.children)
    ]
    return [converted_children, torch.tensor(role, dtype = torch.long)], max_role[0]

In [ ]:
all_src = []
glob_max_role = 1
for i in df["source"].tolist():
    doc = nlp(i.lower())
    tree, max_role = stanza_tree_to_nested(doc.sentences[0].constituency)
    if max_role > glob_max_role:
        glob_max_role = max_role
    if isinstance(tree[0], list):
        all_src.append(tree[0])
    else:
        all_src.append([tree])

In [7]:
test = "This is a test"
doc = nlp(test.lower())
print(doc.sentences[0].constituency)
tree, max_role = stanza_tree_to_nested(doc.sentences[0].constituency)
print(tree)

(ROOT (S (NP (DT this)) (VP (VBZ is) (NP (DT a) (NN test)))))
[[[tensor(0), tensor(0)], [[[tensor(1), tensor(0)], [[[tensor(2), tensor(0)], [tensor(3), tensor(1)]], tensor(1)]], tensor(1)]], tensor(0)]


In [221]:
vocab_size = len(vocab)
print(vocab_size)

3649


In [222]:
from sklearn.model_selection import train_test_split


all_trg = df["context_vector"].tolist()
# random_state=42
# First split: 80% train, 20% temp (which becomes val+test)
train_src, temp_src, train_trg, temp_trg = train_test_split(
    all_src, all_trg, test_size=0.2
)

# Second split: split the 20% temp into 10% val, 10% test
val_src, test_src, val_trg, test_trg = train_test_split(
    temp_src, temp_trg, test_size=0.5
)

print(len(train_src), len(val_src), len(test_src))

800 100 100


In [223]:
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.filler_size, self.role_size, device=self.w.device)
        for pair in x:
            # print(pair)
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            result += torch.outer(filler, role)
        
        result = torch.flatten(result)
        result = self.w @ result
        return result

In [226]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = torch.tensor(train_trg[i], dtype=torch.long).to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [216]:
all_keys = [k for k, v in vocab.items() if v == 1434]
print(all_keys)

# print(vocab[12235])
print(df["source"].iloc[44])

['minnesota']
Fire guts labour accommodation in Ras Al Khaimah


In [232]:
# Train with optimized hyperparams
num_epochs = 50

model = RTPREncode(vocab_size, glob_max_role + 1, 8, 8, model_size).to(device)
learning_rate = 1e-4
norm_val = 5
max_rate = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = torch.tensor(train_trg[i], dtype=torch.long).to(device)
            optimizer.zero_grad()
            try:
                encoding = model(src_batch)
            except TypeError:
                print(src_batch)
                print(i)
            except IndexError:
                print(src_batch)
                print(i)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.000527178718855274
Validation Loss: 0.0004447360327251992
Epoch 2 / 50
Train Loss: 0.0005258199093966237
Validation Loss: 0.00044367779359058004
Epoch 3 / 50
Train Loss: 0.0005248395863429929
Validation Loss: 0.0004430292718139706
Epoch 4 / 50
Train Loss: 0.0005242763717998044
Validation Loss: 0.00044278386047031157
Epoch 5 / 50
Train Loss: 0.0005240059333616285
Validation Loss: 0.00044273124274539223
Epoch 6 / 50
Train Loss: 0.0005238569602535875
Validation Loss: 0.0004427245511062833
Epoch 7 / 50
Train Loss: 0.0005237673777168508
Validation Loss: 0.0004427200470265545
Epoch 8 / 50
Train Loss: 0.0005237186759899799
Validation Loss: 0.0004427146538838652
Epoch 9 / 50
Train Loss: 0.0005236981604612504
Validation Loss: 0.0004427111191906988
Epoch 10 / 50
Train Loss: 0.0005237022900163786
Validation Loss: 0.00044270982603927047
Epoch 11 / 50
Train Loss: 0.0005237310108060602
Validation Loss: 0.0004427109071793809
Epoch 12 / 50
Train Loss: 0.0005237797594214712
V

In [243]:
from sentence_transformers import SentenceTransformer

sent_model = SentenceTransformer('sentence-transformers/paraphrase-MiniLM-L3-v2', device = "mps")

# test = "Minnesota Wild reassign four players to Aeros"
# doc = nlp(test.lower())
# tree, max_role = stanza_tree_to_nested(doc.sentences[0].constituency)
# with torch.no_grad():
#     model.eval()
#     encoding = model(tree[0])
#     output = torch.matmul(encoding, model.wo.T)

#find a model that will succeed with these sorts of things
embedding = sent_model.encode("Alice saw Mary")
embedding2 = sent_model.encode("Mary was seen by Alice")
embedding3 = sent_model.encode("Alice was seen by Mary")

similarities1 = sent_model.similarity(embedding2, embedding)
similarities2 = sent_model.similarity(embedding3, embedding)

print(similarities1)
print(similarities2)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 16584.24it/s]


tensor([[0.8945]])
tensor([[0.8981]])


In [ ]:
def objective(trial):
    num_epochs = 50

    model = RTPREncode(vocab_size, max_role + 1, 8, 8, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        print(f"Validation Loss: {val_loss}")

    return val_loss
    

[tensor(0), tensor(0)]
[[[tensor(1), tensor(0)], [[[tensor(2), tensor(0)], [tensor(3), tensor(1)]], tensor(1)]], tensor(1)]
[tensor(1), tensor(0)]
[[[tensor(2), tensor(0)], [tensor(3), tensor(1)]], tensor(1)]
[tensor(2), tensor(0)]
[tensor(3), tensor(1)]
torch.Size([24])
